[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C27_Model_Compression_Course/01_quantization/01_quantization.ipynb)

# 01 · 整数量化（用 numpy 从零实现）

把权重从浮点压成整数的全套基本功，**每一步都从零写、与全精度对拍**。

**路线**：
1. absmax 对称量化（int8）→ 量化-反量化误差
2. zero-point 非对称量化 → 偏斜分布上完胜对称
3. 量化误差的统计规律：验证 `s²/12`
4. per-tensor / per-channel / group-wise 三种粒度 → 误差随粒度下降
5. int4 + bit packing（打包/解包往返无损）
6. 离群值与 clipping：截断离群值反而降低整体 MSE
7. ✏️ 练习（absmax / zero-point / int4 打包 / 误差度量）
8. 📖 答案 · 🧪 真实 GPT-2 权重胶囊

> **本课纪律**：每个量化都用全精度参考对拍。误差符合理论 → 逻辑正确 → 可迁移到 bitsandbytes/AutoGPTQ。

## 1 · absmax 对称量化（int8）

对称量化令零点 `z=0`，scale 取 absmax：`s = max(|x|)/qmax`，让绝对值最大的元素刚好落到整数边界。
int8 对称用 `qmax=127`（留 −128 不用以保持对称）。量化 `q=round(x/s)`，反量化 `x̂=s·q`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def quantize_absmax(x, bits=8):
    qmax = 2 ** (bits - 1) - 1                 # int8 -> 127
    s = np.abs(x).max() / qmax                 # absmax scale
    q = np.round(x / s).clip(-qmax, qmax).astype(np.int32)
    return q, s

def dequantize_symmetric(q, s):
    return s * q

w = rng.standard_normal(1000) * 0.1            # 模拟一段权重
q, s = quantize_absmax(w, bits=8)
w_hat = dequantize_symmetric(q, s)
err = np.abs(w_hat - w)
print(f'scale s = {s:.6f}')
print(f'量化值范围: [{q.min()}, {q.max()}]  (应在 [-127,127])')
print(f'最大重建误差 = {err.max():.6f}  (应 <= s/2 = {s/2:.6f})')
print(f'相对误差 ‖ŵ-w‖/‖w‖ = {np.linalg.norm(w_hat-w)/np.linalg.norm(w):.4%}')
assert q.min() >= -127 and q.max() <= 127
assert err.max() <= s/2 + 1e-9, '舍入误差不应超过 s/2'
print('✅ absmax 对称量化正确：误差被 s/2 兜住，int8 相对误差很小')

## 2 · zero-point 非对称量化（偏斜分布的救星）

ReLU/GELU 后的激活**全为正、不对称**。对称量化会浪费整个负半轴。非对称量化用 min/max 把任意区间铺满整数范围：
`s=(xmax-xmin)/(qmax-qmin)`，`z=qmin-round(xmin/s)`，量化 `q=round(x/s)+z`，反量化 `x̂=s·(q-z)`。

我们造一段**全正偏斜**的激活，对比对称 vs 非对称的误差。

In [ ]:
def quantize_affine(x, bits=8, signed=False):
    if signed:
        qmin, qmax = -(2**(bits-1)), 2**(bits-1) - 1
    else:
        qmin, qmax = 0, 2**bits - 1               # uint8 -> [0,255]
    xmin, xmax = x.min(), x.max()
    s = (xmax - xmin) / (qmax - qmin)
    z = qmin - np.round(xmin / s)                 # 零点(整数)
    q = (np.round(x / s) + z).clip(qmin, qmax).astype(np.int32)
    return q, s, z

def dequantize_affine(q, s, z):
    return s * (q - z)

# 全正偏斜激活：模拟 ReLU 输出
act = np.abs(rng.standard_normal(5000)) * 2.0     # 全 >= 0

# 非对称(uint8)
q_a, s_a, z_a = quantize_affine(act, bits=8, signed=False)
err_asym = np.linalg.norm(dequantize_affine(q_a, s_a, z_a) - act)
# 对称(把全正数据硬塞对称网格)
q_s, s_s = quantize_absmax(act, bits=8)
err_sym = np.linalg.norm(dequantize_symmetric(q_s, s_s) - act)

print(f'零点 z = {z_a}  (浮点 0 对应的整数; 全正数据 z 接近 0 端)')
print(f'对称  量化误差 = {err_sym:.4f}')
print(f'非对称量化误差 = {err_asym:.4f}')
print(f'非对称把误差降到对称的 {err_asym/err_sym:.1%}')
assert err_asym < err_sym, '偏斜分布上非对称应当更准'
print('✅ 偏斜(全正)分布上，非对称量化用满整个整数范围，误差远小于对称')

## 3 · 量化误差的统计规律：验证 `s²/12`

均匀量化下，单元素误差近似服从 $[-s/2, s/2]$ 均匀分布，期望平方误差约 $s^2/12$。
这是量化的「物理定律」——误差完全由 scale 决定。我们在远离截断边界的数据上验证它。

In [ ]:
def quant_dequant_fixed_s(x, s):
    return s * np.round(x / s)                    # 固定 scale, 不截断

for s in [0.1, 0.05, 0.01]:
    x = rng.uniform(-5, 5, size=500_000)          # 远离边界
    err = quant_dequant_fixed_s(x, s) - x
    mse = (err ** 2).mean()
    theo = s ** 2 / 12
    print(f's={s:5.3f}: 实测MSE={mse:.3e}  理论 s²/12={theo:.3e}  比值={mse/theo:.3f}')
    assert abs(mse - theo) / theo < 0.03, '应在 3% 内'
print('✅ 量化噪声功率 = s²/12，与 scale 平方成正比 —— 这就是为什么要拼命压小 scale')

## 4 · 量化粒度：per-tensor / per-channel / group-wise

粒度越细越准。我们造一个**各行量级差异很大**的权重矩阵（有的行数值大、有的小），
对比三种粒度的**输出误差** `‖Wx-Ŵx‖/‖Wx‖`。预期：per-tensor 最差，group-wise 最好。

In [ ]:
def quant_per_tensor(W, bits=4):
    qmax = 2**(bits-1)-1
    s = np.abs(W).max() / qmax
    return s * np.round(W/s).clip(-qmax, qmax)

def quant_per_channel(W, bits=4):
    qmax = 2**(bits-1)-1
    s = np.abs(W).max(axis=1, keepdims=True) / qmax   # 每行一个 scale
    return s * np.round(W/s).clip(-qmax, qmax)

def quant_group_wise(W, bits=4, g=16):
    qmax = 2**(bits-1)-1
    out = np.empty_like(W)
    for j in range(0, W.shape[1], g):                 # 每行切成每 g 个一组
        blk = W[:, j:j+g]
        s = np.abs(blk).max(axis=1, keepdims=True) / qmax
        out[:, j:j+g] = s * np.round(blk/s).clip(-qmax, qmax)
    return out

d_out, d_in, n = 16, 128, 64
# 各行乘不同幅度，制造量级差异
W = rng.standard_normal((d_out, d_in)) * (10 ** rng.uniform(-1, 1, (d_out, 1)))
X = rng.standard_normal((d_in, n))
def rel_err(Wq): return np.linalg.norm(W@X - Wq@X) / np.linalg.norm(W@X)

e_pt = rel_err(quant_per_tensor(W, 4))
e_pc = rel_err(quant_per_channel(W, 4))
e_gw = rel_err(quant_group_wise(W, 4, g=16))
print(f'int4 per-tensor  相对输出误差 = {e_pt:.3%}')
print(f'int4 per-channel 相对输出误差 = {e_pc:.3%}')
print(f'int4 group-wise  相对输出误差 = {e_gw:.3%}')
assert e_gw < e_pc < e_pt, '误差应随粒度变细单调下降'
print('✅ 粒度越细误差越小：int4 必须靠 per-channel/group-wise 才能保精度')

## 5 · int4 + bit packing：兑现 4× 压缩

numpy 没有 int4 类型。量化到 `[-8,7]` 若还存 int8 容器，显存没省。必须**打包**：两个 int4 拼一个 uint8。

有符号 int4 先加偏移 8 变成 `[0,15]`（无符号）再打包：`packed=(a<<4)|b`；解包 `a=packed>>4`，`b=packed&0xF`，再减 8。

In [ ]:
def pack_int4(q):
    '''q: int 数组, 值域 [-8,7], 长度需为偶数。返回 uint8 打包数组(长度减半)。'''
    assert q.min() >= -8 and q.max() <= 7
    u = (q + 8).astype(np.uint8)                  # [-8,7] -> [0,15]
    a, b = u[0::2], u[1::2]                        # 偶数位放高4位, 奇数位放低4位
    return (a << 4) | b                           # uint8

def unpack_int4(packed):
    '''逆操作: uint8 打包 -> int 数组(值域 [-8,7])。'''
    a = (packed >> 4) & 0xF                        # 高4位
    b = packed & 0xF                              # 低4位
    u = np.empty(packed.size * 2, dtype=np.int32)
    u[0::2], u[1::2] = a, b
    return u - 8                                  # [0,15] -> [-8,7]

q = rng.integers(-8, 8, size=2048)               # 随机 int4 值
packed = pack_int4(q)
restored = unpack_int4(packed)
print(f'原始 {q.size} 个 int4 -> 打包成 {packed.size} 个 uint8 (字节数减半)')
print(f'存储: 若用 int8 需 {q.size} 字节; 打包后仅 {packed.size} 字节 -> {q.size/packed.size:.0f}x')
assert np.array_equal(restored, q), '打包-解包必须往返无损'
assert packed.dtype == np.uint8 and packed.size == q.size // 2
print('✅ 打包/解包往返无损，int4 的 2x(相对int8)/4x(相对fp16) 压缩真正落地')

## 6 · 离群值与 clipping：保护占多数的正常权重

离群值撑大 absmax → scale 变大 → 占 99% 的正常权重精度被浪费。**clipping**：用分位点代替 absmax，
把 scale 压小。代价是少数离群值被夹掉（它们误差变大），但**占绝大多数、真正主导层输出的正常权重精度大涨**。

> 关键：要看 clipping 改善了**哪部分**的误差。它**牺牲少数离群值**、**换正常权重（bulk）的精度**。
> 所以正确的度量是 **bulk（非离群）权重上的 MSE**——这才是主导下游输出的部分。

In [ ]:
# 99% 正常(小) + 1% 重离群值(大)，并记录谁是离群值
normal = rng.standard_normal(9900) * 0.1
outliers = rng.standard_normal(100) * 5.0          # 离群值大 ~50 倍
x = np.concatenate([normal, outliers])
is_bulk = np.concatenate([np.ones(9900, bool), np.zeros(100, bool)])
perm = rng.permutation(x.size); x, is_bulk = x[perm], is_bulk[perm]

def bulk_mse_absmax(x, bits=4):
    q, s = quantize_absmax(x, bits)
    return ((dequantize_symmetric(q, s) - x)[is_bulk] ** 2).mean()  # 只看正常权重

def bulk_mse_clipped(x, bits=4, pct=99.0):
    qmax = 2**(bits-1)-1
    clip_val = np.percentile(np.abs(x), pct)       # 用分位点当范围
    s = clip_val / qmax
    xc = np.clip(x, -clip_val, clip_val)           # 夹掉离群值
    qd = s * np.round(xc/s).clip(-qmax, qmax)
    return ((qd - x)[is_bulk] ** 2).mean()         # 只看正常权重

m_absmax = bulk_mse_absmax(x, bits=4)
m_clip = bulk_mse_clipped(x, bits=4, pct=99.0)
print(f'absmax(被离群值撑大scale): 正常权重 MSE = {m_absmax:.3e}')
print(f'clipping(99分位点)        : 正常权重 MSE = {m_clip:.3e}')
print(f'clipping 把正常权重的 MSE 降到 {m_clip/m_absmax:.2%}')
assert m_clip < m_absmax, 'clipping 应降低正常(bulk)权重的 MSE'
print('✅ 牺牲 1% 离群值，换来 99% 正常权重精度大涨 —— 对付离群值的第一招(更细粒度/GPTQ/AWQ 是后续)')

---
## ✏️ 练习 1：从零写 absmax 对称量化

实现 `my_absmax(x, bits)`，返回 `(q, s)`：对称 absmax 量化。
目标：量化值在 `[-qmax, qmax]`，反量化误差被 `s/2` 兜住。

In [ ]:
def my_absmax(x, bits=8):
    # TODO:
    #   qmax = 2**(bits-1) - 1
    #   s = max(|x|) / qmax
    #   q = round(x/s) 再 clip 到 [-qmax, qmax]
    #   返回 (q.astype(int32), s)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
for bits in [8, 4]:
    x = rng.standard_normal(2000) * 0.3
    q, s = my_absmax(x, bits)
    qmax = 2**(bits-1)-1
    assert q.min() >= -qmax and q.max() <= qmax, f'bits={bits} 越界'
    err = np.abs(s*q - x)
    assert err.max() <= s/2 + 1e-9, f'bits={bits} 误差超过 s/2'
print('✅ 练习 1 通过：absmax 对称量化，误差被 s/2 兜住')

## ✏️ 练习 2：从零写 zero-point 非对称量化

实现 `my_affine(x, bits)`，返回 `(q, s, z)`：用 min/max 的非对称量化（无符号 `[0, 2^bits-1]`）。
目标：在全正偏斜数据上，误差明显小于对称量化。

In [ ]:
def my_affine(x, bits=8):
    # TODO:
    #   qmin, qmax = 0, 2**bits - 1
    #   s = (xmax - xmin) / (qmax - qmin)
    #   z = qmin - round(xmin / s)
    #   q = (round(x/s) + z) 再 clip 到 [qmin, qmax]
    #   返回 (q.astype(int32), s, z)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
act = np.abs(rng.standard_normal(4000)) * 1.5 + 0.5   # 全正、偏离 0
q, s, z = my_affine(act, bits=8)
assert q.min() >= 0 and q.max() <= 255
deq = s * (q - z)
err_asym = np.linalg.norm(deq - act)
qs, ss = my_absmax(act, 8) if False else quantize_absmax(act, 8)
err_sym = np.linalg.norm(ss*qs - act)
assert err_asym < err_sym, '偏斜分布上非对称应更准'
print(f'✅ 练习 2 通过：非对称误差 {err_asym:.3f} < 对称 {err_sym:.3f}')

## ✏️ 练习 3：int4 打包/解包

实现 `my_pack(q)` 与 `my_unpack(packed)`：有符号 int4（`[-8,7]`）的打包与解包。
目标：往返无损，字节数减半。

In [ ]:
def my_pack(q):
    # TODO: q 值域 [-8,7], 偶数长。加偏移 8 变 [0,15]，偶数位<<4 与奇数位 | 起来
    raise NotImplementedError

def my_unpack(packed):
    # TODO: 高4位 = packed>>4 & 0xF, 低4位 = packed & 0xF, 交错还原后减 8
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
q = rng.integers(-8, 8, size=4096)
packed = my_pack(q)
assert packed.dtype == np.uint8 and packed.size == q.size // 2
assert np.array_equal(my_unpack(packed), q), '打包-解包必须往返无损'
print('✅ 练习 3 通过：int4 打包/解包往返无损，字节减半')

## ✏️ 练习 4：量化误差度量

实现 `quant_error_curve(W, X, bits_list)`：对每个 bit 数做 per-channel 量化，返回相对输出误差列表。
目标：bit 越多误差越小（单调递减）。

In [ ]:
def quant_error_curve(W, X, bits_list=(8, 6, 4, 3, 2)):
    # TODO: 对每个 bits 做 per-channel 对称量化(每行一个 absmax scale),
    #       计算 ‖Wx-Ŵx‖/‖Wx‖, 返回与 bits_list 等长的误差 list
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
W = rng.standard_normal((32, 128)) * 0.1
X = rng.standard_normal((128, 50))
bits_list = (8, 6, 4, 3, 2)
errs = quant_error_curve(W, X, bits_list)
assert len(errs) == len(bits_list)
assert all(errs[i] <= errs[i+1] + 1e-9 for i in range(len(errs)-1)), '应单调: bit多->误差小'
for b, e in zip(bits_list, errs):
    print(f'  {b}-bit per-channel: 相对输出误差 {e:.3%}')
print('✅ 练习 4 通过：bit 越多误差越小（精度-压缩率权衡曲线）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_absmax(x, bits=8):
    qmax = 2**(bits-1) - 1
    s = np.abs(x).max() / qmax
    q = np.round(x / s).clip(-qmax, qmax).astype(np.int32)
    return q, s

In [ ]:
# 练习 2 参考答案
def my_affine(x, bits=8):
    qmin, qmax = 0, 2**bits - 1
    xmin, xmax = x.min(), x.max()
    s = (xmax - xmin) / (qmax - qmin)
    z = qmin - np.round(xmin / s)
    q = (np.round(x / s) + z).clip(qmin, qmax).astype(np.int32)
    return q, s, z

In [ ]:
# 练习 3 参考答案
def my_pack(q):
    u = (q + 8).astype(np.uint8)
    return (u[0::2] << 4) | u[1::2]

def my_unpack(packed):
    a = (packed >> 4) & 0xF; b = packed & 0xF
    u = np.empty(packed.size * 2, dtype=np.int32)
    u[0::2], u[1::2] = a, b
    return u - 8

In [ ]:
# 练习 4 参考答案
def quant_error_curve(W, X, bits_list=(8, 6, 4, 3, 2)):
    errs = []
    base = np.linalg.norm(W @ X)
    for bits in bits_list:
        qmax = 2**(bits-1) - 1
        s = np.abs(W).max(axis=1, keepdims=True) / qmax
        Wq = s * np.round(W/s).clip(-qmax, qmax)
        errs.append(np.linalg.norm(W@X - Wq@X) / base)
    return errs

---
## 🧪 真实数据胶囊：量化真实 GPT-2 的权重

用**真实 GPT-2** 的一个权重张量来量化，看真实权重分布上的效果。
**联网失败会自动回退**到统计匹配的合成权重（同形状、近似分布），算法与结论不变——离线也能完整跑。

任务：实现 `quantize_weight_int8_perchannel(W)`，对真实权重做 per-channel int8 量化，验证相对输出误差 < 1%。

In [ ]:
def load_gpt2_mlp_weight():
    '''尝试加载真实 GPT-2 的一个 MLP 权重; 失败则回退到统计匹配的合成权重。'''
    try:
        from transformers import GPT2Model
        m = GPT2Model.from_pretrained('gpt2')
        W = m.h[0].mlp.c_fc.weight.detach().numpy().astype(np.float64)
        print(f'[真实 GPT-2] c_fc 权重 shape={W.shape}')
        return W
    except Exception as e:
        print(f'[回退合成] 联网/transformers 不可用 ({type(e).__name__})，用统计匹配的合成权重')
        rng2 = np.random.default_rng(42)
        # GPT-2 c_fc: (768, 3072), 权重近似零均值正态, std~0.1, 含少量离群值
        W = rng2.standard_normal((768, 3072)) * 0.1
        idx = rng2.integers(0, W.size, size=W.size//500)
        W.flat[idx] *= 8.0                       # 注入少量离群值
        return W

W_gpt2 = load_gpt2_mlp_weight()
print(f'权重统计: mean={W_gpt2.mean():.4f}, std={W_gpt2.std():.4f}, absmax={np.abs(W_gpt2).max():.4f}')

In [ ]:
def quantize_weight_int8_perchannel(W):
    # TODO: per-channel(每行一个 absmax scale) int8 对称量化, 返回反量化后的 Ŵ
    raise NotImplementedError

In [ ]:
# 自测
rng3 = np.random.default_rng(1)
X = rng3.standard_normal((W_gpt2.shape[1], 64))
W_hat = quantize_weight_int8_perchannel(W_gpt2)
rel = np.linalg.norm(W_gpt2@X - W_hat@X) / np.linalg.norm(W_gpt2@X)
comp_ratio = 2.0  # fp16(2B) -> int8(1B)
print(f'真实(或合成)GPT-2 权重 per-channel int8:')
print(f'  相对输出误差 = {rel:.4%}  压缩率 = {comp_ratio:.0f}x (vs fp16)')
assert rel < 0.02, f'int8 应近乎无损, 实际 {rel:.4%}'
print('✅ 胶囊通过：真实权重上 per-channel int8 近乎无损（~1%），2x 压缩白拿')
print('   (含离群值的层误差略高；模块 02 的 GPTQ/AWQ 能把它再压一截)')

In [ ]:
# 📖 胶囊参考答案
def quantize_weight_int8_perchannel(W):
    qmax = 127
    s = np.abs(W).max(axis=1, keepdims=True) / qmax
    q = np.round(W / s).clip(-qmax, qmax)
    return s * q

### 小结
- **仿射量化** `q=round(x/s)+z`：scale 定分辨率、zero-point 定偏移。对称(z=0,absmax)用于权重，非对称(min/max)用于偏斜激活。
- **粒度**：per-tensor < per-channel < group-wise，越细越准。int8 用 per-channel 近乎无损；**int4 必须 group-wise**。
- **量化噪声 = s²/12**：误差完全由 scale 决定，而 scale 被离群值撑大 → **离群值是头号敌人**（clipping/更细粒度/模块02 的算法来对付）。
- **bit packing**：两个 int4 拼一个字节，才真正兑现 4× 压缩（否则白存）。
- **weight-only 量化**省的是**带宽/显存**不是算力 → 访存受限的 LLM 推理直接加速（即便推理时要反量化）。

下一站：**模块 02 · GPTQ 与 AWQ** —— RTN 不是最优；用逆 Hessian 误差补偿和激活感知缩放，把 int4 做到近乎无损。